In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Добавляем корень проекта в sys.path, чтобы импортировать src.*
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.split import make_time_folds, expanding_window_splits

# Пути
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
REPORTS_FIGURES = PROJECT_ROOT / "reports" / "figures"

# Настройки визуализации
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (10, 5)

# Загружаем train
train = pd.read_parquet(DATA_INTERIM / "train.parquet", engine="pyarrow")
print(f"Train: {train.shape}")
print(f"Память: {train.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

In [ ]:
# Строим фолды по TransactionDT
folds = make_time_folds(train["TransactionDT"], n_splits=5)
train["fold"] = folds

# Проверяем размеры фолдов
print("Распределение транзакций по фолдам:")
fold_sizes = pd.Series(folds).value_counts().sort_index()
for f, n in fold_sizes.items():
    print(f"  Fold {f}: {n:>7,} транзакций")
print()

# Границы времени для каждого фолда
print("Временные границы фолдов (в днях):")
train["day"] = (train["TransactionDT"] // 86400).astype(np.int16)
for f in range(5):
    mask = folds == f
    day_min = train.loc[mask, "day"].min()
    day_max = train.loc[mask, "day"].max()
    fraud_rate = train.loc[mask, "isFraud"].mean() * 100
    print(f"  Fold {f}: дни {day_min}..{day_max} "
          f"({day_max - day_min + 1} дней), fraud rate {fraud_rate:.2f}%")

In [ ]:
# Готовим данные для картинки: на оси Y — номер фолда, на оси X — день
fig, ax = plt.subplots(figsize=(12, 4))

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
for f in range(5):
    mask = folds == f
    day_min = train.loc[mask, "day"].min()
    day_max = train.loc[mask, "day"].max()
    ax.barh(y=f, width=day_max - day_min + 1, left=day_min,
            height=0.6, color=colors[f], alpha=0.7,
            label=f"Fold {f}")

ax.set_yticks(range(5))
ax.set_yticklabels([f"Fold {i}" for i in range(5)])
ax.invert_yaxis()  # Fold 0 сверху
ax.set_xlabel("День от начала датасета")
ax.set_title("Временные границы фолдов (все фолды равны по количеству транзакций)",
             fontsize=12)
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "06_time_folds.png", bbox_inches="tight")
plt.show()

In [ ]:
# Колонки, которые НЕ должны быть фичами
drop_cols = ["TransactionID", "isFraud", "TransactionDT", "fold", "day"]

# Target и фичи
y = train["isFraud"].astype(np.int8)
X = train.drop(columns=drop_cols)

print(f"X: {X.shape}")
print(f"y: {y.shape}, fraud rate: {y.mean() * 100:.2f}%")
print()

# Проверяем типы
print("Распределение типов:")
print(X.dtypes.value_counts())
print()

# Список категориальных колонок — понадобится LightGBM для правильной обработки
cat_cols = X.select_dtypes(include=["category"]).columns.tolist()
print(f"Категориальных колонок: {len(cat_cols)}")
print(f"Первые 5: {cat_cols[:5]}")

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import time

# Генерируем пары (train, valid) через expanding window
splits = expanding_window_splits(folds, n_splits=5)
print(f"Количество фолдов для обучения: {len(splits)}")

# Минимальные гиперпараметры baseline — намеренно осторожные
lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1,
    "random_state": 42,
}

# Массив для OOF-предсказаний (заполним по мере обучения каждого фолда)
oof_preds = np.zeros(len(X), dtype=np.float32)
cv_scores = []
models = []  # сохраним обученные модели для дальнейшего предсказания на test

for fold_num, (train_idx, valid_idx) in enumerate(splits, start=1):
    t0 = time.time()

    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[valid_idx], y.iloc[valid_idx]

    train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
    valid_data = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols,
                              reference=train_data)

    model = lgb.train(
        params=lgb_params,
        train_set=train_data,
        num_boost_round=2000,
        valid_sets=[valid_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0),  # молчать
        ],
    )

    # OOF-предсказания для этого фолда
    oof_preds[valid_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    fold_auc = roc_auc_score(y_va, oof_preds[valid_idx])
    cv_scores.append(fold_auc)
    models.append(model)

    elapsed = time.time() - t0
    print(f"Fold {fold_num}: AUC = {fold_auc:.5f} "
          f"(best_iter={model.best_iteration}, time={elapsed:.0f}s)")

print()
print(f"CV AUC: {np.mean(cv_scores):.5f} ± {np.std(cv_scores):.5f}")

# OOF AUC на тех строках, которые побывали в валидации (fold 1-4, без fold 0)
valid_mask = folds > 0
print(f"OOF AUC (only validated rows): {roc_auc_score(y[valid_mask], oof_preds[valid_mask]):.5f}")

In [ ]:
# Загружаем test
test = pd.read_parquet(DATA_INTERIM / "test.parquet", engine="pyarrow")
print(f"Test: {test.shape}")

# ПАТЧ: переименовываем id-XX -> id_XX
rename_map = {c: c.replace("id-", "id_") for c in test.columns if c.startswith("id-")}
test = test.rename(columns=rename_map)
print(f"Переименовано колонок: {len(rename_map)}")

# Готовим фичи
test_drop = ["TransactionID", "TransactionDT"]
X_test = test.drop(columns=test_drop)

# Выравниваем колонки в том же порядке, что в X
X_test = X_test[X.columns]
print(f"X_test: {X_test.shape}")

# Приводим категориальные dtypes из train к test
# (иначе LightGBM может не узнать категории)
for col in cat_cols:
    if col in X_test.columns:
        X_test[col] = X_test[col].astype(X[col].dtype)

# Предсказание = среднее по 4 моделям
test_preds = np.zeros(len(X_test), dtype=np.float32)
for i, model in enumerate(models):
    p = model.predict(X_test, num_iteration=model.best_iteration)
    test_preds += p / len(models)
    print(f"  Model {i+1}: predicted, min={p.min():.4f}, max={p.max():.4f}, mean={p.mean():.4f}")

print()
print(f"Final: min={test_preds.min():.4f}, max={test_preds.max():.4f}, "
      f"mean={test_preds.mean():.4f}")

In [ ]:
DATA_PREDICTIONS = PROJECT_ROOT / "data" / "predictions"
DATA_PREDICTIONS.mkdir(parents=True, exist_ok=True)

# Submission для Kaggle
submission = pd.DataFrame({
    "TransactionID": test["TransactionID"],
    "isFraud": test_preds,
})

submission_path = DATA_PREDICTIONS / "sub_lgbm_baseline_v0.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved: {submission_path.name}")
print(f"Rows: {len(submission)}, size: {submission_path.stat().st_size / 1024:.0f} KB")
print()
print(submission.head())

# Сохраним OOF для будущих ансамблей
oof_df = pd.DataFrame({
    "TransactionID": train["TransactionID"],
    "isFraud_true": y,
    "isFraud_pred": oof_preds,
    "fold": folds,
})
oof_path = DATA_PREDICTIONS / "oof_lgbm_baseline_v0.csv"
oof_df.to_csv(oof_path, index=False)
print(f"OOF saved: {oof_path.name}")